In [1]:
import pandas as pd
import numpy as np

df_freq = pd.read_csv("../../data/freMTPL2freq.csv")
df_sev = pd.read_csv("../../data/df_sev_clean.csv")

# ۱. ادغام دو جدول
claims_per_policy = df_sev.groupby('IDpol')['ClaimAmount'].sum().reset_index()
df_merged = pd.merge(df_freq, claims_per_policy, on='IDpol', how='left')
df_merged['ClaimAmount'] = df_merged['ClaimAmount'].fillna(0)

# ۲. تعریف گروه‌های سن راننده (DrivAge Groups)
df_merged['DrivAge_Group'] = pd.cut(
    df_merged['DrivAge'], 
    bins=[17, 25, 35, 55, 70, 100], 
    labels=['18-25 (Young)', '26-35', '36-55 (Middle)', '56-70', '70+ (Senior)']
)

# ۳. تعریف گروه‌های سن خودرو (VehAge Groups)
df_merged['VehAge_Group'] = pd.cut(
    df_merged['VehAge'], 
    bins=[-1, 1, 4, 9, 100], 
    labels=['0-1 (New)', '2-4', '5-9', '10+ (Old)']
)

# ۴. تعریف گروه‌های ضریب جریمه-تخفیف (BonusMalus Groups)
df_merged['BM_Group'] = pd.cut(
    df_merged['BonusMalus'], 
    bins=[0, 50, 75, 100, 350], 
    labels=['<=50 (Max Bonus)', '51-75', '76-100', '>100 (Malus)']
)

# تابع کمکی برای محاسبه نرخ‌ها بر اساس گروه‌های مختلف
def analyze_segment(df, group_col):
    summary = df.groupby(group_col, observed=False).agg(
        Total_Exposure=('Exposure', 'sum'),
        Total_Claims=('ClaimNb', 'sum'),
        Total_Cost=('ClaimAmount', 'sum')
    ).reset_index()
    
    summary['Claim_Frequency'] = summary['Total_Claims'] / summary['Total_Exposure']
    summary['Average_Severity'] = summary['Total_Cost'] / summary['Total_Claims']
    summary['Average_Severity'] = summary['Average_Severity'].fillna(0)
    return summary

# تحلیل دسته‌های مختلف
driv_age_analysis = analyze_segment(df_merged, 'DrivAge_Group')
veh_age_analysis = analyze_segment(df_merged, 'VehAge_Group')
bm_analysis = analyze_segment(df_merged, 'BM_Group')
fuel_analysis = analyze_segment(df_merged, 'VehGas')

print("--- Driver Age Patterns ---")
print(driv_age_analysis)

print("--- veh_age_analysis ---")
print(veh_age_analysis)

print("--- fuel_analysis ---")
print(fuel_analysis)

print("\n--- BonusMalus Patterns ---")
print(bm_analysis)


# قوی‌ترین الگوها:
#
# سن راننده:
# قوی‌ترین الگو در بین رانندگان جوان (۱۸ تا ۲۵ سال) مشاهده می‌شود؛
# این گروه هم بالاترین فراوانی خسارت (۱۴.۸٪) و هم بالاترین
# میانگین شدت خسارت (۵۱۲۱.۶۲) را دارند.
# این نشان می‌دهد که خسارت‌های مرتبط با این گروه سنی هم بیشتر
# و هم به طور متوسط پرهزینه‌تر هستند.
#
# سن خودرو:
# خودروهای قدیمی‌تر (۱۰+ سال) بالاترین میانگین شدت خسارت (۲۶۹۸.۰۸)
# را نشان می‌دهند، اگرچه فراوانی خسارت آن‌ها بالاترین نیست.
# این نشان می‌دهد که خسارت‌های مربوط به خودروهای قدیمی‌تر
# به طور متوسط گران‌تر هستند.
#
# امتیاز-خسارت (Bonus-Malus):
# فراوانی خسارت در گروه‌های Bonus-Malus به شدت افزایش می‌یابد؛
# از ۵.۱۵٪ برای مقادیر کمتر یا مساوی ۵۰
# تا ۳۵.۱۷٪ برای مقادیر بالاتر از ۱۰۰.
# گروه ۷۶ تا ۱۰۰ نیز بالاترین میانگین شدت خسارت (۳۳۶۹.۹۳) را دارد.
#
# نوع سوخت:
# خودروهای با سوخت معمولی نسبت به خودروهای دیزلی،
# فراوانی کمتری در خسارت‌ها دارند (۶.۹۲٪ در مقابل ۷.۸۸٪)،
# اما میانگین شدت خسارت آن‌ها بالاتر است
# (۲۴۸۶.۸۸ در مقابل ۲۰۵۱.۶۵).
#
# به طور کلی، واضح‌ترین الگوها شامل فراوانی و شدت بالای خسارت‌ها
# در بین رانندگان جوان و افزایش شدید فراوانی خسارت با مقادیر بالاتر
# Bonus-Malus هستند.
#
# این نتایج فقط ارتباط و الگوهای موجود در داده‌ها را نشان می‌دهند
# و ثابت نمی‌کنند که این عوامل باعث ایجاد خسارت می‌شوند.

--- Driver Age Patterns ---
    DrivAge_Group  Total_Exposure  Total_Claims   Total_Cost  Claim_Frequency  \
0   18-25 (Young)    16226.113263          2403  12307261.54         0.148095   
1           26-35    69265.528075          5167  10607248.80         0.074597   
2  36-55 (Middle)   176961.742696         12937  24445506.13         0.073106   
3           56-70    69074.076804          4327   8156579.57         0.062643   
4    70+ (Senior)    26955.374625          1610   4392620.46         0.059728   

   Average_Severity  
0       5121.623612  
1       2052.883453  
2       1889.580747  
3       1885.042655  
4       2728.335689  
--- veh_age_analysis ---
  VehAge_Group  Total_Exposure  Total_Claims   Total_Cost  Claim_Frequency  \
0    0-1 (New)    49465.206348          3509   7485825.24         0.070939   
1          2-4    79546.287576          5904  12187778.85         0.074221   
2          5-9   100251.297795          8044  15987956.00         0.080238   
3    10+ (Old)  